# WFCR — Análise de benchmarks

Este notebook produz as tabelas e gráficos agregados a partir dos trials salvos em um banco
SQLite. Escolha o banco e use os filtros por tamanho da instância e por algoritmo.


In [ ]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

# Onde procurar os bancos SQLite (glob "*.db").
DB_DIRS = [Path("..") / "databases", Path("..")]
OUT_DIR = Path("..") / "images"

COLORS = ["#d62728", "#1f77b4", "#2ca02c", "#9467bd", "#17becf", "#bcbd22", "#8c564b", "#e377c2", "#7f7f7f"]


def has_flow_schema(path) -> bool:
    try:
        conn = sqlite3.connect(path)
        try:
            cols = [row[1] for row in conn.execute("PRAGMA table_info(edges)")]
        finally:
            conn.close()
        return "flow" in cols
    except sqlite3.Error:
        return False

DB_FILES = [p for d in DB_DIRS for p in sorted(d.glob("*.db")) if has_flow_schema(p)]


def load_trials(db, instance=None):
    conn = sqlite3.connect(db)
    query = "SELECT algorithm, instance, instance_size, duration_ms, total_cost FROM trials"
    params = []
    if instance:
        query += " WHERE instance = ?"
        params.append(instance)
    return pd.read_sql_query(query, conn, params=params)


def summary(df):
    g = (
        df.groupby("algorithm")
        .agg(
            best=("total_cost", "min"),
            avg=("total_cost", "mean"),
            median=("total_cost", "median"),
            std=("total_cost", "std"),
            runs=("total_cost", "size"),
        )
        .reset_index()
        .sort_values(["best", "avg"])
    )
    for col in ["best", "avg", "median"]:
        g[col] = g[col] / 1e6
    g["std"] = g["std"].fillna(0.0) / 1e6
    return g.rename(
        columns={
            "best": "Melhor (M)",
            "avg": "Média (M)",
            "median": "Mediana (M)",
            "std": "Desvio (M)",
            "runs": "Execuções",
        }
    ).round(2)

In [6]:
def _gap_data(df):
    med = df.groupby(["instance", "algorithm"])["total_cost"].median().reset_index()
    pivot = med.pivot(index="instance", columns="algorithm", values="total_cost")

    baseline = pivot.sum().idxmin()
    pivot = pivot.sort_values(baseline)
    base = np.nan_to_num(pivot[baseline].to_numpy(), nan=0.0)

    gaps = {}
    for algo in pivot.columns:
        cost = pivot[algo].to_numpy()
        gaps[algo] = np.where(base > 0, (cost - base) / base * 100.0, 0.0)
    return baseline, gaps


def gap_figure(df, sorted=False):
    baseline, gaps = _gap_data(df)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.axhline(0, color="black", lw=3, label=f"{baseline} (Baseline)")

    color_idx = 0
    for algo, vals in gaps.items():
        if algo == baseline:
            continue
        color = COLORS[color_idx % len(COLORS)]
        color_idx += 1
        vals = np.sort(vals) if sorted else vals
        ax.plot(
            np.arange(vals.size),
            np.clip(vals, -5, 5),
            color=color,
            lw=2,
            marker=None if sorted else "o",
            ms=4,
            label=algo,
        )

    ax.set_xlabel("Instâncias (Ordenadas pelo GAP)" if sorted else "Instâncias (Ordenadas por custo)")
    ax.set_ylabel("Diferença para a Baseline (%)")
    ax.set_xticks([])
    ax.set_ylim(-10, 10)
    ax.legend(loc="upper left")
    fig.tight_layout()
    return fig

In [7]:
def scalability(df, out=None):
    g = (
        df.groupby(["algorithm", "instance_size"])
        .agg(avg_cost=("total_cost", "mean"), avg_time=("duration_ms", "mean"))
        .reset_index()
    )
    if g.empty:
        return

    sizes = sorted(g["instance_size"].unique())
    algos = sorted(g["algorithm"].unique())
    x = np.arange(len(sizes))
    n = len(algos)
    width = 0.8 / n

    specs = [
        ("avg_time", "Tempo Médio (ms)", 1.0, "scalability_time.png"),
        ("avg_cost", "Custo Médio (M)", 1e6, "scalability_cost.png"),
    ]

    for col, ylabel, scale, fname in specs:
        fig, ax = plt.subplots(figsize=(12, 8))
        for i, algo in enumerate(algos):
            vals = []
            for s in sizes:
                row = g[(g["algorithm"] == algo) & (g["instance_size"] == s)]
                vals.append(float(row[col].iloc[0]) / scale if not row.empty else 0.0)
            pos = x + (i - (n - 1) / 2) * width
            bars = ax.bar(pos, vals, width * 0.85, label=algo, color=COLORS[i % len(COLORS)])
            ax.bar_label(bars, fmt="%.1f", padding=2, fontsize=8)
        ax.set_xticks(x)
        ax.set_xticklabels([str(s) for s in sizes])
        ax.set_xlabel("Quantidade de Turbinas")
        ax.set_ylabel(ylabel)
        ax.legend(loc="upper left")
        fig.tight_layout()
        plt.show()
        if out:
            fig.savefig(Path(out) / fname, dpi=150)
        plt.close(fig)

In [8]:
import ipywidgets as widgets
from IPython.display import display

db_w = widgets.Dropdown(
    options=[str(p) for p in DB_FILES],
    value=str(DB_FILES[0]) if DB_FILES else None,
    description="Banco",
    layout=widgets.Layout(width="55%"),
)
size_w = widgets.SelectMultiple(description="Tamanho", rows=8)
algo_w = widgets.SelectMultiple(description="Algoritmo", rows=8)
out = widgets.Output()

df_all = None
_updating = False


def load_db():
    global df_all, _updating
    df_all = load_trials(db_w.value)
    _updating = True
    sizes = sorted(df_all["instance_size"].unique().tolist())
    size_w.options = sizes
    size_w.value = sizes
    algos = sorted(df_all["algorithm"].unique())
    algo_w.options = algos
    algo_w.value = algos
    _updating = False


def render(*_):
    if _updating or df_all is None:
        return
    out.clear_output()
    d = df_all[
        df_all["instance_size"].isin(size_w.value) & df_all["algorithm"].isin(algo_w.value)
    ]
    with out:
        if d.empty:
            print("Nenhum dado com os filtros selecionados.")
            return
        display(summary(d))
        print("GAP relativo à baseline")
        gap_figure(d)
        plt.show()
        print("GAP relativo ordenado")
        gap_figure(d, sorted=True)
        plt.show()
        print("Escalabilidade")
        scalability(d, OUT_DIR)


def on_db_change(*_):
    load_db()
    render()


db_w.observe(on_db_change, names="value")
size_w.observe(render, names="value")
algo_w.observe(render, names="value")

display(widgets.VBox([db_w, widgets.HBox([size_w, algo_w]), out]))
on_db_change()